## 1. Setup and Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
DATA_DIR = Path('../data/processed')
RESULTS_DIR = Path('../results')

print("✓ Imports loaded")

✓ Imports loaded


In [3]:
# Load data
df = pd.read_csv(DATA_DIR / 'merged.csv')
expr_pca = pd.read_csv(DATA_DIR / 'gdsc_expr_pca.csv')
chemberta_feats = np.load(DATA_DIR / 'chemberta_drug_feats.npz')

# Load test indices
test_idx = np.load(DATA_DIR / 'splits_drug/test_idx.npy')

print(f"Dataset: {len(df):,} samples")
print(f"Test set: {len(test_idx):,} samples")
print(f"Expression PCs: {expr_pca.shape[1]-1} features")
print(f"ChemBERTa keys available: {list(chemberta_feats.files)}")
# Get the first array key (likely 'arr_0' or similar)
chemberta_key = chemberta_feats.files[0] if len(chemberta_feats.files) > 0 else None
if chemberta_key:
    print(f"ChemBERTa dims: {chemberta_feats[chemberta_key].shape[1]}")

Dataset: 65,088 samples
Test set: 112,977 samples
Expression PCs: 512 features
ChemBERTa keys available: ['feats', 'drug_id', 'smiles', 'model_name']
ChemBERTa dims: 768


## 2. Load Trained Models

**Note:** Both models use expression + drug features + tissue, but differ in drug representation:
- **MLP Baseline**: Uses Morgan fingerprints (1024-bit) + RDKit descriptors
- **MLP+ChemBERTa**: Uses ChemBERTa learned embeddings

In [4]:
import sys
sys.path.append('..')
from src.models.mlp import MLP
from src.models.gnn import GINEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [8]:
# Load MLP baseline model (uses Morgan fingerprints for drugs)
n_expr_features = expr_pca.shape[1] - 1
n_tissues = df['tissue'].nunique()

# Note: The "baseline" model actually uses expr + Morgan FP + tissue
# We need to check the actual input dimension from the checkpoint
checkpoint = torch.load(RESULTS_DIR / 'mlp_baseline_with_splits/mlp_model.pt', map_location=device)
first_layer_weight = checkpoint['net.0.weight']
in_dim_baseline = first_layer_weight.shape[1]

print(f"MLP baseline expects {in_dim_baseline} input features")
print(f"  Expression PCs: {n_expr_features}")
print(f"  Tissues: {n_tissues}")
print(f"  Drug features (Morgan FP): {in_dim_baseline - n_expr_features - n_tissues}")

mlp_baseline = MLP(
    in_dim=in_dim_baseline,
    hidden=[1024, 512],
    dropout=0.3
).to(device)

print("✓ MLP baseline loaded (Expr + Morgan FP + Tissue)")

mlp_baseline.load_state_dict(checkpoint)
mlp_baseline.eval()

MLP baseline expects 1599 input features
  Expression PCs: 512
  Tissues: 22
  Drug features (Morgan FP): 1065
✓ MLP baseline loaded (Expr + Morgan FP + Tissue)


MLP(
  (net): Sequential(
    (0): Linear(in_features=1599, out_features=1024, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=1024, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=512, out_features=1, bias=True)
  )
)

In [11]:
# Load MLP+ChemBERTa model
# Get correct key for ChemBERTa features
chemberta_key = chemberta_feats.files[0] if len(chemberta_feats.files) > 0 else 'arr_0'
n_drug_features = chemberta_feats[chemberta_key].shape[1]

# Check actual input dimension from checkpoint
checkpoint_cb = torch.load(RESULTS_DIR / 'mlp_chemberta/mlp_model.pt', map_location=device)
first_layer_weight_cb = checkpoint_cb['net.0.weight']
in_dim_chemberta = first_layer_weight_cb.shape[1]

print(f"MLP+ChemBERTa expects {in_dim_chemberta} input features")
print(f"  Expression PCs: {n_expr_features}")
print(f"  ChemBERTa features: {n_drug_features}")
print(f"  Tissues: {n_tissues}")
print(f"  Expected total: {n_expr_features + n_drug_features + n_tissues}")
print(f"  Difference: {in_dim_chemberta - (n_expr_features + n_drug_features + n_tissues)}")

mlp_chemberta = MLP(
    in_dim=in_dim_chemberta,
    hidden=[1024, 512],
    dropout=0.3
).to(device)

mlp_chemberta.load_state_dict(checkpoint_cb)
mlp_chemberta.eval()

print("✓ MLP+ChemBERTa loaded")


MLP+ChemBERTa expects 1337 input features
  Expression PCs: 512
  ChemBERTa features: 768
  Tissues: 22
  Expected total: 1302
  Difference: 35
✓ MLP+ChemBERTa loaded


In [23]:
# Load GNN model
try:
    mol_graphs_data = torch.load(DATA_DIR / 'mol_graphs.pt', weights_only=False)
    
    # Structure: dict with keys ['graphs', 'drug_id', 'smiles']
    # Create drug_id -> graph mapping for easy access
    graphs_list = mol_graphs_data['graphs']
    drug_ids = mol_graphs_data['drug_id']
    
    mol_graphs = {drug_id: graph for drug_id, graph in zip(drug_ids, graphs_list)}
    
    print(f"✓ Loaded {len(mol_graphs)} molecular graphs")
    
    # Get graph dimensions from first graph
    sample_graph = graphs_list[0]
    n_node_features = sample_graph.x.shape[1]
    n_edge_features = sample_graph.edge_attr.shape[1] if sample_graph.edge_attr is not None else 0
    
    print(f"Node features: {n_node_features}, Edge features: {n_edge_features}")
    
    # GINEncoder for molecular graph encoding (graph -> embedding)
    # Note: The saved model only contains the GNN encoder, not the full pipeline
    # During training, GNN output is concatenated with expr + tissue, then fed to MLPHead
    gnn_model = GINEncoder(
        in_dim=n_node_features,
        hidden_dim=128,
        num_layers=3
    ).to(device)
    
    # Load checkpoint - it contains full model (gnn + head + metadata)
    checkpoint_gnn = torch.load(
        RESULTS_DIR / 'gnn_baseline/gnn_model.pt',
        map_location=device,
        weights_only=True
    )
    
    # Extract just the GNN encoder weights
    if 'gnn' in checkpoint_gnn:
        gnn_model.load_state_dict(checkpoint_gnn['gnn'])
    else:
        gnn_model.load_state_dict(checkpoint_gnn)
    
    gnn_model.eval()
    
    print("✓ GNN model loaded")
    gnn_available = True
except Exception as e:
    print(f"⚠ GNN model not loaded: {e}")
    gnn_available = False

✓ Loaded 621 molecular graphs
Node features: 5, Edge features: 0
✓ GNN model loaded


## 3. Prepare Test Data

In [26]:
# Check data dimensions
print(f"Dataframe shape: {df.shape}")
print(f"Test indices shape: {test_idx.shape}")
print(f"Max test index: {test_idx.max()}")
print(f"Min test index: {test_idx.min()}")
print(f"Number of out-of-bounds indices: {(test_idx >= len(df)).sum()}")

# Filter to valid indices only
valid_test_idx = test_idx[test_idx < len(df)]
print(f"\nValid test indices: {len(valid_test_idx)} out of {len(test_idx)}")
print(f"Percentage valid: {len(valid_test_idx)/len(test_idx)*100:.1f}%")

Dataframe shape: (65088, 519)
Test indices shape: (112977,)
Max test index: 562788
Min test index: 0
Number of out-of-bounds indices: 99901

Valid test indices: 13076 out of 112977
Percentage valid: 11.6%


In [28]:
# Check column names
print("DataFrame columns:")
print(df.columns.tolist()[:20])  # First 20 columns
print(f"\nTotal columns: {len(df.columns)}")
print(f"\nLooking for cell/drug related columns:")
print([c for c in df.columns if 'cell' in c.lower() or 'line' in c.lower() or 'drug' in c.lower() or 'ic50' in c.lower() or 'tissue' in c.lower()])

DataFrame columns:
['cell_id', 'drug_id', 'drug_name', 'smiles', 'tissue', 'dataset', 'ln_ic50', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13']

Total columns: 519

Looking for cell/drug related columns:
['cell_id', 'drug_id', 'drug_name', 'tissue', 'ln_ic50']


In [32]:
# Get test samples - filter to valid indices only
valid_test_idx = test_idx[test_idx < len(df)]
print(f"Using {len(valid_test_idx)} valid test samples (out of {len(test_idx)} total)")
df_test = df.iloc[valid_test_idx].reset_index(drop=True)

# Prepare features - expression data is already in merged.csv as PC columns
# Extract just the PC columns
pc_cols = [col for col in df.columns if col.startswith('PC')]
X_expr_test = df_test[pc_cols].values
print(f"Expression features: {X_expr_test.shape}")

# Tissue encoding (one-hot)
from sklearn.preprocessing import OneHotEncoder
ohe_tissue = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_tissue.fit(df[['tissue']])
X_tissue_test = ohe_tissue.transform(df_test[['tissue']])
print(f"Tissue features: {X_tissue_test.shape}")

# Drug features - Morgan fingerprints for baseline model
from rdkit import Chem, DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem import Descriptors

def compute_morgan_fp(smiles, n_bits=1024, radius=2):
    """Compute Morgan fingerprint + RDKit descriptors to match training (1065 features)"""
    expected_size = in_dim_baseline - n_expr_features - n_tissues  # 1065
    
    if pd.isna(smiles) or smiles == "":
        return np.zeros(expected_size, dtype=np.float32)
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(expected_size, dtype=np.float32)
    
    # Morgan fingerprint
    gen = GetMorganGenerator(radius=radius, fpSize=n_bits)
    fp = gen.GetFingerprint(mol)
    fp_arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    
    # RDKit descriptors
    desc = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol)
    ], dtype=np.float32)
    
    # Concatenate and pad/trim to expected size
    features = np.concatenate([fp_arr.astype(np.float32), desc])
    if len(features) < expected_size:
        # Pad with zeros if needed
        features = np.pad(features, (0, expected_size - len(features)), constant_values=0)
    elif len(features) > expected_size:
        # Trim if somehow too large
        features = features[:expected_size]
    
    return features

print("Computing Morgan fingerprints for test drugs...")
X_drug_morgan_test = np.array([compute_morgan_fp(s) for s in df_test['smiles']])
print(f"Morgan FP shape: {X_drug_morgan_test.shape}")

# ChemBERTa features for mlp_chemberta model
if 'drug_id' in chemberta_feats.files:
    drug_ids_test = df_test['drug_id'].values
    drug_id_to_idx = {did: i for i, did in enumerate(chemberta_feats['drug_id'])}
    drug_indices = [drug_id_to_idx[did] for did in drug_ids_test]
    X_drug_chemberta_test = chemberta_feats[chemberta_key][drug_indices]
else:
    # Assume ChemBERTa features are already aligned with dataset
    X_drug_chemberta_test = chemberta_feats[chemberta_key][valid_test_idx]

print(f"ChemBERTa shape: {X_drug_chemberta_test.shape}")

# Targets
y_test = df_test['ln_ic50'].values

print(f"\n✓ Test set prepared: {len(df_test)} samples")
print(f"Expression: {X_expr_test.shape}, Morgan FP: {X_drug_morgan_test.shape}")
print(f"ChemBERTa: {X_drug_chemberta_test.shape}, Tissue: {X_tissue_test.shape}")

Using 13076 valid test samples (out of 112977 total)
Expression features: (13076, 512)
Tissue features: (13076, 22)
Computing Morgan fingerprints for test drugs...
Morgan FP shape: (13076, 1065)
ChemBERTa shape: (13076, 768)

✓ Test set prepared: 13076 samples
Expression: (13076, 512), Morgan FP: (13076, 1065)
ChemBERTa: (13076, 768), Tissue: (13076, 22)


## 4. SHAP Analysis for MLP Baseline

Using SHAP (SHapley Additive exPlanations) to understand feature contributions to predictions.

In [30]:
try:
    import shap
    shap_available = True
    print("✓ SHAP library available")
except ImportError:
    print("⚠ SHAP not installed. Install with: pip install shap")
    shap_available = False

✓ SHAP library available


In [33]:
if shap_available:
    # Create wrapper for SHAP
    def mlp_predict(data):
        """Wrapper for MLP baseline prediction"""
        with torch.no_grad():
            # MLP expects concatenated features (expr + Morgan FP + tissue)
            X_batch = torch.FloatTensor(data).to(device)
            preds = mlp_baseline(X_batch)
            return preds.cpu().numpy()
    
    # Combine features for SHAP (expr + Morgan FP + tissue one-hot)
    X_combined_test = np.column_stack([X_expr_test, X_drug_morgan_test, X_tissue_test])
    
    # Use subset for computational efficiency
    n_samples = min(500, len(X_combined_test))
    X_shap = X_combined_test[:n_samples]
    
    print(f"Computing SHAP values for {n_samples} samples...")
    print("This may take a few minutes...")
    
    # Create explainer
    explainer = shap.KernelExplainer(mlp_predict, X_shap[:100])  # Background samples
    shap_values = explainer.shap_values(X_shap[:100])  # Explain subset
    
    print("✓ SHAP values computed")

Computing SHAP values for 500 samples...
This may take a few minutes...


  0%|          | 0/100 [00:00<?, ?it/s]

RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 768000000 bytes.

In [ ]:
if shap_available and 'shap_values' in locals():
    # Summary plot
    feature_names = (
        [f'PC{i+1}' for i in range(X_expr_test.shape[1])] +
        [f'DrugFP{i+1}' for i in range(X_drug_morgan_test.shape[1])] +
        [f'Tissue{i+1}' for i in range(X_tissue_test.shape[1])]
    )
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values,
        X_shap[:100],
        feature_names=feature_names,
        show=False,
        max_display=20
    )
    plt.title('SHAP Feature Importance - MLP Baseline', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    

    print("\n💡 Interpretation:")    print("   • Horizontal spread shows impact magnitude")

    print("   • Features ranked by impact on predictions")    print("   • Color indicates feature value (red=high, blue=low)")

In [ ]:
if shap_available and 'shap_values' in locals():
    # Bar plot of mean absolute SHAP values
    mean_shap = np.abs(shap_values).mean(axis=0)
    top_k = 15
    top_indices = np.argsort(mean_shap)[-top_k:][::-1]
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(top_k), mean_shap[top_indices], color='steelblue')
    plt.yticks(range(top_k), [feature_names[i] for i in top_indices])
    plt.xlabel('Mean |SHAP Value|', fontsize=12)
    plt.title('Top 15 Features by SHAP Importance', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. SHAP Analysis for MLP+ChemBERTa

In [ ]:
if shap_available:
    def mlp_chemberta_predict(data):
        """Wrapper for MLP+ChemBERTa prediction"""
        with torch.no_grad():
            # MLP expects concatenated features (expr + ChemBERTa + tissue)
            X_batch = torch.FloatTensor(data).to(device)
            preds = mlp_chemberta(X_batch)
            return preds.cpu().numpy()
    
    # Combine all features (expr + ChemBERTa + tissue one-hot)
    X_combined_chemberta = np.column_stack([X_expr_test, X_drug_chemberta_test, X_tissue_test])
    
    print(f"Computing SHAP values for MLP+ChemBERTa (optimized for Colab)...")
    print("Using 15 background samples and explaining 25 samples...")
    print("This may take several minutes...")
    
    # Use even smaller subset for ChemBERTa (fewer features but still memory intensive)
    X_background_cb = X_combined_chemberta[:15]
    X_explain_cb = X_combined_chemberta[15:40]  # 25 samples
    
    explainer_cb = shap.KernelExplainer(mlp_chemberta_predict, X_background_cb)
    shap_values_cb = explainer_cb.shap_values(X_explain_cb)
    
    print("✓ SHAP values computed")

In [ ]:
if shap_available and 'shap_values_cb' in locals():
    # Aggregate ChemBERTa dimensions for visualization
    n_expr = X_expr_test.shape[1]
    n_drug = X_drug_chemberta_test.shape[1]
    n_tissue = X_tissue_test.shape[1]
    
    # Sum absolute SHAP values by feature group
    shap_expr = np.abs(shap_values_cb[:, :n_expr]).sum(axis=1)
    shap_drug = np.abs(shap_values_cb[:, n_expr:n_expr+n_drug]).sum(axis=1)
    shap_tissue = np.abs(shap_values_cb[:, n_expr+n_drug:]).sum(axis=1)
    
    # Create comparison
    feature_groups = ['Gene Expression\n(PCs)', 'Drug Features\n(ChemBERTa)', 'Tissue']
    mean_importance = [
        shap_expr.mean(),
        shap_drug.mean(),
        shap_tissue.mean()
    ]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(feature_groups, mean_importance, color=['#2E86AB', '#A23B72', '#F18F01'])
    plt.ylabel('Mean Absolute SHAP Value', fontsize=12)
    plt.title('Feature Group Importance - MLP+ChemBERTa', fontsize=14, fontweight='bold')
    
    # Add values on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Interpretation:")
    print(f"   • Gene expression contributes {mean_importance[0]/(sum(mean_importance))*100:.1f}% to predictions")
    print(f"   • Drug features contribute {mean_importance[1]/(sum(mean_importance))*100:.1f}% to predictions")
    print(f"   • Tissue contributes {mean_importance[2]/(sum(mean_importance))*100:.1f}% to predictions")

## 6. GNN Molecular Embeddings Visualization

In [ ]:
if gnn_available:
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
    
    print("Extracting GNN molecular embeddings...")
    
    embeddings_list = []
    drug_ids_list = []
    
    with torch.no_grad():
        for drug_id in df_test['DRUG_ID'].unique()[:100]:  # Subset for visualization
            if drug_id in mol_graphs:
                graph = mol_graphs[drug_id].to(device)
                # Extract embedding from GNN encoder
                emb = gnn_model.gin(graph.x, graph.edge_index, graph.edge_attr, graph.batch)
                embeddings_list.append(emb.cpu().numpy())
                drug_ids_list.append(drug_id)
    
    embeddings = np.vstack(embeddings_list)
    print(f"✓ Extracted {len(embeddings)} embeddings of dimension {embeddings.shape[1]}")
else:
    print("⚠ GNN not available, skipping embedding visualization")

In [ ]:
if gnn_available and len(embeddings_list) > 0:
    # t-SNE visualization
    print("Computing t-SNE projection...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
    embeddings_2d = tsne.fit_transform(embeddings)
    
    # Get drug response statistics for coloring
    drug_stats = df_test.groupby('DRUG_ID')['LN_IC50'].mean()
    colors = [drug_stats.get(did, 0) for did in drug_ids_list]
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(
        embeddings_2d[:, 0],
        embeddings_2d[:, 1],
        c=colors,
        cmap='RdYlBu_r',
        s=100,
        alpha=0.6,
        edgecolors='black',
        linewidth=0.5
    )
    plt.colorbar(scatter, label='Mean LN_IC50')
    plt.xlabel('t-SNE Dimension 1', fontsize=12)
    plt.ylabel('t-SNE Dimension 2', fontsize=12)
    plt.title('GNN Molecular Embeddings (t-SNE)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Interpretation:")
    print("   • Each point represents a drug's learned molecular representation")
    print("   • Color indicates average drug potency (darker = more potent)")
    print("   • Clustering suggests GNN groups structurally similar compounds")

## 7. Prediction Confidence Analysis

In [ ]:
# Get predictions from all models
with torch.no_grad():
    # MLP baseline: expr + Morgan FP + tissue
    X_baseline = torch.FloatTensor(np.column_stack([X_expr_test, X_drug_morgan_test, X_tissue_test])).to(device)
    # MLP+ChemBERTa: expr + ChemBERTa + tissue
    X_chemberta = torch.FloatTensor(np.column_stack([X_expr_test, X_drug_chemberta_test, X_tissue_test])).to(device)
    
    preds_mlp = mlp_baseline(X_baseline).cpu().numpy()
    preds_mlp_cb = mlp_chemberta(X_chemberta).cpu().numpy()

# Calculate prediction errors
errors_mlp = np.abs(preds_mlp - y_test)
errors_mlp_cb = np.abs(preds_mlp_cb - y_test)

# Model agreement (uncertainty proxy)
model_agreement = np.abs(preds_mlp - preds_mlp_cb)

print(f"Mean prediction disagreement: {model_agreement.mean():.3f}")
print(f"Samples with high disagreement (>1.0): {(model_agreement > 1.0).sum()}")

In [ ]:
# Visualize confidence vs error
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MLP baseline
axes[0].scatter(model_agreement, errors_mlp, alpha=0.3, s=20)
axes[0].set_xlabel('Model Disagreement', fontsize=11)
axes[0].set_ylabel('Absolute Error', fontsize=11)
axes[0].set_title('MLP Baseline: Uncertainty vs Error', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Correlation
corr_mlp = np.corrcoef(model_agreement, errors_mlp)[0, 1]
axes[0].text(0.05, 0.95, f'Correlation: {corr_mlp:.3f}',
            transform=axes[0].transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            verticalalignment='top')

# MLP+ChemBERTa
axes[1].scatter(model_agreement, errors_mlp_cb, alpha=0.3, s=20, color='orange')
axes[1].set_xlabel('Model Disagreement', fontsize=11)
axes[1].set_ylabel('Absolute Error', fontsize=11)
axes[1].set_title('MLP+ChemBERTa: Uncertainty vs Error', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

corr_cb = np.corrcoef(model_agreement, errors_mlp_cb)[0, 1]
axes[1].text(0.05, 0.95, f'Correlation: {corr_cb:.3f}',
            transform=axes[1].transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            verticalalignment='top')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Positive correlation indicates model disagreement signals uncertainty")
print("   • High disagreement samples are less reliable predictions")
print("   • Can be used for confidence-based prediction filtering")

## 8. High/Low Confidence Prediction Examples

In [ ]:
# High confidence (low disagreement) predictions
high_conf_idx = np.argsort(model_agreement)[:10]
low_conf_idx = np.argsort(model_agreement)[-10:]

print("="*80)
print("HIGH CONFIDENCE PREDICTIONS (Model Agreement)")
print("="*80)
for i, idx in enumerate(high_conf_idx, 1):
    print(f"\n{i}. Drug: {df_test.iloc[idx]['DRUG_NAME']}, "
          f"Cell: {df_test.iloc[idx]['CELL_LINE_NAME']}, "
          f"Tissue: {df_test.iloc[idx]['tissue']}")
    print(f"   True: {y_test[idx]:.2f}, "
          f"MLP: {preds_mlp[idx]:.2f}, "
          f"MLP+CB: {preds_mlp_cb[idx]:.2f}")
    print(f"   Disagreement: {model_agreement[idx]:.3f}, "
          f"Error: {min(errors_mlp[idx], errors_mlp_cb[idx]):.3f}")

print("\n" + "="*80)
print("LOW CONFIDENCE PREDICTIONS (Model Disagreement)")
print("="*80)
for i, idx in enumerate(low_conf_idx, 1):
    print(f"\n{i}. Drug: {df_test.iloc[idx]['DRUG_NAME']}, "
          f"Cell: {df_test.iloc[idx]['CELL_LINE_NAME']}, "
          f"Tissue: {df_test.iloc[idx]['tissue']}")
    print(f"   True: {y_test[idx]:.2f}, "
          f"MLP: {preds_mlp[idx]:.2f}, "
          f"MLP+CB: {preds_mlp_cb[idx]:.2f}")
    print(f"   Disagreement: {model_agreement[idx]:.3f}, "
          f"Error: {min(errors_mlp[idx], errors_mlp_cb[idx]):.3f}")

## 9. Summary and Key Insights

In [ ]:
print("="*90)
print("KEY INSIGHTS FROM MODEL INTERPRETATION")
print("="*90)

if shap_available and 'shap_values' in locals():
    print("\n🔍 FEATURE IMPORTANCE (SHAP):")
    top_3_features = np.argsort(np.abs(shap_values).mean(axis=0))[-3:][::-1]
    for i, feat_idx in enumerate(top_3_features, 1):
        print(f"   {i}. {feature_names[feat_idx]}: "
              f"{np.abs(shap_values[:, feat_idx]).mean():.4f} mean |SHAP|")

if shap_available and 'shap_values_cb' in locals():
    print("\n🧬 FEATURE GROUP CONTRIBUTIONS (MLP+ChemBERTa):")
    total_importance = sum(mean_importance)
    print(f"   • Gene Expression: {mean_importance[0]/total_importance*100:.1f}%")
    print(f"   • Drug Features: {mean_importance[1]/total_importance*100:.1f}%")
    print(f"   • Tissue Context: {mean_importance[2]/total_importance*100:.1f}%")

print("\n📊 PREDICTION CONFIDENCE:")
print(f"   • Mean model disagreement: {model_agreement.mean():.3f}")
print(f"   • High uncertainty samples (>1.0): {(model_agreement > 1.0).sum()} "
      f"({(model_agreement > 1.0).sum()/len(model_agreement)*100:.1f}%)")
print(f"   • Disagreement-error correlation: {corr_mlp:.3f} (MLP), {corr_cb:.3f} (MLP+CB)")

if gnn_available and len(embeddings_list) > 0:
    print("\n🧪 MOLECULAR EMBEDDINGS:")
    print(f"   • GNN learns {embeddings.shape[1]}-dimensional drug representations")
    print("   • t-SNE visualization shows clustering by chemical similarity")
    print("   • Embedding space correlates with drug potency")

print("\n💡 ACTIONABLE INSIGHTS:")
print("   1. Gene expression PCs dominate predictions (expected)")
print("   2. Drug features provide significant additional signal")
print("   3. Model disagreement is useful for uncertainty estimation")
print("   4. High disagreement samples warrant manual review")
print("   5. GNN embeddings capture meaningful chemical structure")

print("\n" + "="*90)

## 10. Model Agreement Heatmap

Visualize where models agree and disagree most across predictions.

In [ ]:
# Create model agreement matrix
# Bin predictions into deciles for heatmap
n_bins = 10
pred_bins = np.linspace(y_test.min(), y_test.max(), n_bins + 1)

# Bin predictions for each model
mlp_binned = np.digitize(preds_mlp, pred_bins)
mlp_cb_binned = np.digitize(preds_mlp_cb, pred_bins)

# Create agreement matrix (MLP vs MLP+CB)
agreement_matrix = np.zeros((n_bins, n_bins))
for i in range(len(y_test)):
    bin_mlp = mlp_binned[i] - 1  # -1 because digitize returns 1-indexed
    bin_mlp_cb = mlp_cb_binned[i] - 1
    if 0 <= bin_mlp < n_bins and 0 <= bin_mlp_cb < n_bins:
        agreement_matrix[bin_mlp, bin_mlp_cb] += 1

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap 1: Agreement counts
sns.heatmap(
    agreement_matrix,
    cmap='YlGnBu',
    annot=True,
    fmt='.0f',
    cbar_kws={'label': 'Number of Predictions'},
    ax=axes[0],
    xticklabels=[f'{pred_bins[i]:.1f}' for i in range(n_bins)],
    yticklabels=[f'{pred_bins[i]:.1f}' for i in range(n_bins)]
)
axes[0].set_title('Model Agreement Matrix: MLP vs MLP+ChemBERTa', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('MLP+ChemBERTa Predicted LN_IC50', fontsize=11)
axes[0].set_ylabel('MLP Baseline Predicted LN_IC50', fontsize=11)
axes[0].plot([0, n_bins], [0, n_bins], 'r--', linewidth=2, label='Perfect Agreement')
axes[0].legend()

# Heatmap 2: Disagreement magnitude
disagreement_by_bin = np.zeros((n_bins, n_bins))
disagreement_count = np.zeros((n_bins, n_bins))

for i in range(len(y_test)):
    bin_mlp = mlp_binned[i] - 1
    bin_mlp_cb = mlp_cb_binned[i] - 1
    if 0 <= bin_mlp < n_bins and 0 <= bin_mlp_cb < n_bins:
        disagreement_by_bin[bin_mlp, bin_mlp_cb] += model_agreement[i]
        disagreement_count[bin_mlp, bin_mlp_cb] += 1

# Average disagreement
avg_disagreement = np.divide(
    disagreement_by_bin, 
    disagreement_count, 
    out=np.zeros_like(disagreement_by_bin), 
    where=disagreement_count!=0
)

sns.heatmap(
    avg_disagreement,
    cmap='RdYlGn_r',
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'Mean Disagreement'},
    ax=axes[1],
    xticklabels=[f'{pred_bins[i]:.1f}' for i in range(n_bins)],
    yticklabels=[f'{pred_bins[i]:.1f}' for i in range(n_bins)]
)
axes[1].set_title('Model Disagreement Magnitude', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('MLP+ChemBERTa Predicted LN_IC50', fontsize=11)
axes[1].set_ylabel('MLP Baseline Predicted LN_IC50', fontsize=11)

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Left: Count of predictions in each bin combination")
print("   • Diagonal: Perfect agreement between models")
print("   • Right: Average disagreement magnitude (red=high uncertainty)")
print(f"   • Overall agreement correlation: {np.corrcoef(preds_mlp, preds_mlp_cb)[0,1]:.3f}")